# Sistema Híbrido de Recomendaciones: SVD + K-Means

**Objetivo:** Recomendar productos según estado del usuario:
- **Cold-start (0 eventos)**: Cluster-based recommendations
- **Warm-start light (1-5 eventos)**: Blend cluster + SVD
- **Warm-start (>5 eventos)**: Pure SVD collaborative filtering

**Input:**
- `data/final/events_final.csv` — Viene de pipelineV_3.0.ipynb, contiene eventos de usuario-producto.

**Inputs de clustering (generados por otros notebooks):**
- `data/final/user_cluster_map.csv` — de `clustering_analisis.ipynb`
- `data/final/recomendaciones_cold_start.csv` — de `clustering_analisis.ipynb`
- `data/final/user_clustering_gmm_results.csv` — de `Notebook_Clustering_NMM.ipynb` 

**Output:**
- Modelo SVD entrenado
- Mapeo de usuarios a tier (0, 1, 2)
- Función de recomendación híbrida


In [5]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import json
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42

print("=== IMPORTACIONES HECHAS ===")

=== IMPORTACIONES HECHAS ===


---
## Sección 1: Cargar Clustering & Datos de Eventos

In [6]:
from pathlib import Path

_here = Path(os.path.abspath(""))
_project_root = next(
    p for p in [_here, _here.parent, _here.parent.parent]
    if (p / "data" / "final").exists()
)

events_path = _project_root / "data" / "final" / "events_final.csv"

df_events = pd.read_csv(events_path)
df_events['event_time'] = pd.to_datetime(df_events['event_time'], utc=True)

# ── Pesos por tipo de evento ─────────────────────────────────────────────────
weight_map = {'view': 1, 'cart': 2, 'purchase': 3}
df_events['peso'] = df_events['event_type'].map(weight_map).fillna(0)

# ── Interacciones con peso simple ────────────────────────────────────────────
df_interactions = (
    df_events
    .groupby(['user_id', 'product_id'], as_index=False)['peso']
    .sum()
)
df_interactions = df_interactions[df_interactions['peso'] > 0].reset_index(drop=True)

# ── Interacciones con score temporal (peso ajustado por recencia) ────────────
# score_temporal = peso * exp(-0.01 * dias_desde_ultima_interaccion)
# Interacciones recientes pesan mas que las antiguas
fecha_ref = df_events['event_time'].max()
ultima_interaccion = (
    df_events.groupby(['user_id', 'product_id'])['event_time']
    .max()
    .reset_index()
    .rename(columns={'event_time': 'ultima_interaccion'})
)
df_interactions_temporal = df_interactions.merge(ultima_interaccion, on=['user_id', 'product_id'])
df_interactions_temporal['dias'] = (fecha_ref - df_interactions_temporal['ultima_interaccion']).dt.days
df_interactions_temporal['score_temporal'] = (
    df_interactions_temporal['peso'] * np.exp(-0.01 * df_interactions_temporal['dias'])
).round(4)
df_interactions_temporal = df_interactions_temporal[df_interactions_temporal['score_temporal'] > 0].reset_index(drop=True)

print(f"Largo de los eventos:                    {df_events.shape}")
print(f"Largo de las interacciones (peso):       {df_interactions.shape}")
print(f"Largo de las interacciones (temporal):   {df_interactions_temporal.shape}")

# ── Contar eventos por usuario para determinar tier ─────────────────────────
user_event_count = df_events.groupby('user_id').size().reset_index(name='event_count')

def assign_tier(n):
    if n <= 5:
        return 1
    return 2

user_event_count['tier'] = user_event_count['event_count'].apply(assign_tier)

print("\nDISTRIBUCION DE USUARIOS POR TIER")
print(f"Tier 1 (1-5 eventos):  {(user_event_count['tier']==1).sum():,}")
print(f"Tier 2 (>5 eventos):   {(user_event_count['tier']==2).sum():,}")


Largo de los eventos:                    (884312, 17)
Largo de las interacciones (peso):       (556637, 3)
Largo de las interacciones (temporal):   (556637, 6)

DISTRIBUCION DE USUARIOS POR TIER
Tier 1 (1-5 eventos):  382,745
Tier 2 (>5 eventos):   24,492


---
## Sección 2: Construir Modelo SVD en Matriz User-Item

In [7]:
N_COMPONENTS = 50

# Crear mapeos de IDs (iguales para ambos modelos)
unique_users    = sorted(df_interactions['user_id'].unique())
unique_products = sorted(df_interactions['product_id'].unique())
user_id_map     = {uid: idx for idx, uid in enumerate(unique_users)}
product_id_map  = {pid: idx for idx, pid in enumerate(unique_products)}

def construir_matriz(df, columna_score):
    """Construye una matriz sparse usuario-producto con la columna de score indicada."""
    row  = df['user_id'].map(user_id_map).values
    col  = df['product_id'].map(product_id_map).values
    data = df[columna_score].values
    return csr_matrix((data, (row, col)), shape=(len(unique_users), len(unique_products)))

# ── Modelo A: peso simple ─────────────────────────────────────────────────────
matriz_peso = construir_matriz(df_interactions, 'peso')
svd_peso    = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE, n_iter=100)
latent_peso = svd_peso.fit_transform(matriz_peso)

# ── Modelo B: score temporal ──────────────────────────────────────────────────
df_temporal_alineado = df_interactions_temporal[
    df_interactions_temporal['user_id'].isin(unique_users) &
    df_interactions_temporal['product_id'].isin(unique_products)
].copy()
matriz_temporal = construir_matriz(df_temporal_alineado, 'score_temporal')
svd_temporal    = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE, n_iter=100)
latent_temporal = svd_temporal.fit_transform(matriz_temporal)

# ── Comparacion ───────────────────────────────────────────────────────────────
print(f"Matriz user-item: {matriz_peso.shape}")
print(f"Sparsidad: {1 - matriz_peso.nnz / (matriz_peso.shape[0] * matriz_peso.shape[1]):.4f}")
print(f"\n{'Modelo':<30} {'Varianza explicada':>20}")
print(f"{'SVD con peso simple':<30} {svd_peso.explained_variance_ratio_.sum():>20.4f}")
print(f"{'SVD con score temporal':<30} {svd_temporal.explained_variance_ratio_.sum():>20.4f}")
print(f"\nDiferencia: {(svd_temporal.explained_variance_ratio_.sum() - svd_peso.explained_variance_ratio_.sum()):+.4f}")

# Usar el mejor modelo como svd_model y user_latent para el resto del notebook
if svd_temporal.explained_variance_ratio_.sum() >= svd_peso.explained_variance_ratio_.sum():
    svd_model   = svd_temporal
    user_latent = latent_temporal
    print("\n Se usa SVD con score temporal")
else:
    svd_model   = svd_peso
    user_latent = latent_peso
    print("\n Se usa SVD con peso simple")


Matriz user-item: (407237, 53452)
Sparsidad: 1.0000

Modelo                           Varianza explicada
SVD con peso simple                          0.2767
SVD con score temporal                       0.3732

Diferencia: +0.0965

 Se usa SVD con score temporal


---
## Sección 3: Definir Tiers y Crear Diccionarios de Mapeo

In [9]:
# Cargar mapeo usuario -> cluster generado por Clustering_Kmeans.ipynb
user_cluster_map_path = _project_root / 'data' / 'final' / 'user_cluster_map.csv'
if not user_cluster_map_path.exists():
    raise FileNotFoundError(
        f'No existe {user_cluster_map_path}. '
        'Ejecuta Clustering_Kmeans.ipynb primero y asegurate de correr la celda de exportacion.'
    )

df_user_cluster = pd.read_csv(user_cluster_map_path)
user_cluster_map = dict(zip(df_user_cluster['user_id'], df_user_cluster['cluster_id']))
print(f'Usuarios con cluster asignado: {len(user_cluster_map):,}')

# Construir user_profile: user_id -> (tier, cluster_id, latent_factors)
user_profile = {}

for user_id in unique_users:
    tier_row = user_event_count[user_event_count['user_id'] == user_id]['tier'].values
    tier = int(tier_row[0]) if len(tier_row) > 0 else 0
    cluster = user_cluster_map.get(user_id, -1)
    user_idx = user_id_map[user_id]
    latent_factors = user_latent[user_idx]

    user_profile[user_id] = {
        'tier': tier,
        'cluster_id': cluster,
        'latent_factors': latent_factors,
    }

print(f'User profiles creados: {len(user_profile):,}')

# Cargar recomendaciones cold-start por cluster
cold_start_path = _project_root / 'data' / 'final' / 'recomendaciones_cold_start.csv'
if cold_start_path.exists():
    cold_start_recs = pd.read_csv(cold_start_path)
    print(f'Recomendaciones cold-start cargadas: {cold_start_recs.shape[0]} filas')
else:
    raise FileNotFoundError(
        f'No existe {cold_start_path}. '
        'Ejecuta Clustering_Kmeans.ipynb primero.'
    )


Usuarios con cluster asignado: 407,237
User profiles creados: 407,237
Recomendaciones cold-start cargadas: 80 filas


In [ ]:
# Cargar clasificacion GMM (activo=0 / pasivo=1) por usuario
gmm_results_path = _project_root / 'data' / 'final' / 'user_clustering_gmm_results.csv'
if gmm_results_path.exists():
    df_gmm_map = pd.read_csv(gmm_results_path, usecols=['user_id', 'gmm_cluster'])
    user_gmm_map = dict(zip(df_gmm_map['user_id'], df_gmm_map['gmm_cluster']))
    n_activos = sum(v == 0 for v in user_gmm_map.values())
    n_pasivos = sum(v == 1 for v in user_gmm_map.values())
    print(f'GMM clusters cargados: {len(user_gmm_map):,} usuarios')
    print(f'  Activos  (0): {n_activos:,} ({n_activos / len(user_gmm_map) * 100:.1f}%)')
    print(f'  Pasivos  (1): {n_pasivos:,} ({n_pasivos / len(user_gmm_map) * 100:.1f}%)')
else:
    user_gmm_map = {}
    print('!!!!!!!!!! No existe user_clustering_gmm_results.csv.')
    print('   Tier 1 usara pesos fijos 0.7/0.3 (modo pasivo por defecto).')
    print('   Sino ejecuta Notebook_Clustering_NMM.ipynb para generarlo.')

!!!!!!!!!! No existe user_clustering_gmm_results.csv.
   Tier 1 usara pesos fijos 0.7/0.3 (modo pasivo por defecto).
   Sino ejecuta Notebook_Clustering_NMM.ipynb para generarlo.


---
## Sección 4: Función de Recomendación Híbrida

In [ ]:
def recomendar_nuevo_usuario(n=10):
    """Fallback para usuarios que no existen en ningun dataset (primera sesion real)."""
    cluster_default = int(cold_start_recs['cluster_id'].value_counts().idxmax())
    recs = cold_start_recs[cold_start_recs['cluster_id'] == cluster_default].head(n).copy()
    recs = recs[['product_id', 'score_cluster']].rename(columns={'score_cluster': 'score'})
    recs['method'] = 'global-popular'
    return recs.reset_index(drop=True)


def recomendar(user_id, n=10):
    """
    Recomendacion hibrida por tier.

    Tier 0: sin historial  -> recomendaciones por cluster (cold start)
    Tier 1: 1-5 eventos    -> hibrido cluster + SVD, pesos ajustados por GMM
    Tier 2: >5 eventos     -> SVD puro
    """
    if user_id not in user_profile:
        return recomendar_nuevo_usuario(n=n)

    profile = user_profile[user_id]
    tier = profile['tier']
    cluster_id = profile['cluster_id']
    latent_factors = profile['latent_factors']

    # ── TIER 0: sin historial ──────────────────────────────────────────────
    if tier == 0:
        if cluster_id < 0:
            return recomendar_nuevo_usuario(n=n)
        recs = cold_start_recs[cold_start_recs['cluster_id'] == cluster_id].head(n).copy()
        recs = recs[['product_id', 'score_cluster']].rename(columns={'score_cluster': 'score'})
        recs['method'] = 'cluster-based'
        return recs.reset_index(drop=True)

    # ── TIER 1: hibrido con pesos ajustados por GMM ────────────────────────
    elif tier == 1:
        gmm_cluster = user_gmm_map.get(user_id, 1)  # default pasivo si no esta en el mapa
        alfa, beta = (0.4, 0.6) if gmm_cluster == 0 else (0.7, 0.3)
        tipo = 'activo' if gmm_cluster == 0 else 'pasivo'

        cluster_recs = cold_start_recs[cold_start_recs['cluster_id'] == cluster_id].copy()
        product_latent = svd_model.components_.T
        svd_df = pd.DataFrame({
            'product_id': unique_products,
            'svd_score': product_latent @ latent_factors,
        })

        blended = cluster_recs[['product_id', 'score_cluster']].merge(svd_df, on='product_id', how='outer')
        blended['score_cluster'] = blended['score_cluster'].fillna(0)
        blended['svd_score']     = blended['svd_score'].fillna(0)

        c_min, c_max = blended['score_cluster'].min(), blended['score_cluster'].max()
        s_min, s_max = blended['svd_score'].min(),     blended['svd_score'].max()
        blended['score_cluster_norm'] = (blended['score_cluster'] - c_min) / (c_max - c_min + 1e-6)
        blended['svd_score_norm']     = (blended['svd_score']     - s_min) / (s_max - s_min + 1e-6)

        blended['score'] = alfa * blended['score_cluster_norm'] + beta * blended['svd_score_norm']
        blended = blended.sort_values('score', ascending=False).head(n)
        blended['method'] = f'hybrid(a={alfa},b={beta},{tipo})'
        return blended[['product_id', 'score', 'method']].reset_index(drop=True)

    # ── TIER 2: SVD puro ───────────────────────────────────────────────────
    else:
        product_latent = svd_model.components_.T
        recs = pd.DataFrame({
            'product_id': unique_products,
            'score': product_latent @ latent_factors,
        })
        recs = recs.sort_values('score', ascending=False).head(n)
        recs['method'] = 'svd'
        return recs[['product_id', 'score', 'method']].reset_index(drop=True)


print("Funciones de recomendacion listas!")

Funciones de recomendacion listas!


---
## Sección 5: Testear Pipeline de Recomendaciones

In [ ]:
print("TESTING RECOMENDACIONES POR TIER \n")

# Tier 0
tier0_users = user_event_count[user_event_count['tier'] == 0]['user_id'].head(1).values
if len(tier0_users) > 0:
    print(f"TIER 0 (Cold-start): user_id={tier0_users[0]}")
    print(recomendar(tier0_users[0], n=10))
    print()

# Tier 1
tier1_users = user_event_count[user_event_count['tier'] == 1]['user_id'].head(1).values
if len(tier1_users) > 0:
    print(f"TIER 1 (Warm-light): user_id={tier1_users[0]}")
    print(recomendar(tier1_users[0], n=10))
    print()

# Tier 2
tier2_users = user_event_count[user_event_count['tier'] == 2]['user_id'].head(1).values
if len(tier2_users) > 0:
    print(f"TIER 2 (Warm): user_id={tier2_users[0]}")
    print(recomendar(tier2_users[0], n=10))
    print()

# Usuario completamente nuevo (no en ninguna tabla)
print("USUARIO NUEVO (no en sistema):")
print(recomendar_nuevo_usuario(n=10))

TESTING RECOMENDACIONES POR TIER 

TIER 1 (Warm-light): user_id=1515915625353226922
   product_id     score                      method
0     1785245  0.703622  hybrid(a=0.7,b=0.3,pasivo)
1     3642540  0.417656  hybrid(a=0.7,b=0.3,pasivo)
2      246841  0.331173  hybrid(a=0.7,b=0.3,pasivo)
3     3828501  0.283124  hybrid(a=0.7,b=0.3,pasivo)
4     1785246  0.227999  hybrid(a=0.7,b=0.3,pasivo)
5      802811  0.179664  hybrid(a=0.7,b=0.3,pasivo)
6      194199  0.177947  hybrid(a=0.7,b=0.3,pasivo)
7      471387  0.142979  hybrid(a=0.7,b=0.3,pasivo)
8      775032  0.135623  hybrid(a=0.7,b=0.3,pasivo)
9     1674260  0.119552  hybrid(a=0.7,b=0.3,pasivo)

TIER 2 (Warm): user_id=1515915625353230683
   product_id     score method
0      893192  0.001511    svd
1     4100254  0.000486    svd
2     4102053  0.000131    svd
3     3791773  0.000119    svd
4     4183856  0.000112    svd
5     3661135  0.000105    svd
6     4079422  0.000084    svd
7     1571204  0.000067    svd
8     3647921  0.0000

---
## Sección 6: Exportar Modelo y Mapeos para Producción

In [ ]:
# Exportar modelo SVD y mapeos para producción
_models_dir = _project_root / 'models'
_models_dir.mkdir(exist_ok=True)

# 1. Guardar modelo SVD
svd_model_path = _models_dir / 'svd_model.pkl'
with open(svd_model_path, 'wb') as f:
    pickle.dump(svd_model, f)
print(f" SVD model guardado: {svd_model_path}")

# 2. Guardar user_profile
user_profile_path = _models_dir / 'user_profile.pkl'
with open(user_profile_path, 'wb') as f:
    pickle.dump(user_profile, f)
print(f" User profiles guardados: {user_profile_path}")

# 3. Guardar mapeos de IDs
id_maps = {
    'user_id_map': user_id_map,
    'product_id_map': product_id_map,
    'reverse_product_id_map': {v: k for k, v in product_id_map.items()}
}
id_maps_path = _models_dir / 'id_maps.pkl'
with open(id_maps_path, 'wb') as f:
    pickle.dump(id_maps, f)
print(f" ID maps guardados: {id_maps_path}")

# 4. Guardar tier distribution
tier_dist = user_event_count.groupby('tier').size().to_dict()
tier_info = {
    'tier_distribution': tier_dist,
    'total_users': len(user_event_count),
    'n_components': N_COMPONENTS,
    'variance_explained': float(svd_model.explained_variance_ratio_.sum())
}
with open(_models_dir / 'model_info.json', 'w') as f:
    json.dump(tier_info, f, indent=2)
print(f" Model info guardada: {_models_dir / 'model_info.json'}")

# 5. Exportar user-tier mapping como CSV para referencia rápida
user_tier_export = user_event_count[['user_id', 'event_count', 'tier']].copy()
user_tier_export['cluster_id'] = user_tier_export['user_id'].map(user_cluster_map)
user_tier_export.to_csv(_project_root / 'data' / 'final' / 'user_tier_mapping.csv', index=False)
print(f" User-tier mapping guardada: {_project_root / 'data' / 'final' / 'user_tier_mapping.csv'}")


 SVD model guardado: ../models/svd_model.pkl
 User profiles guardados: ../models/user_profile.pkl
 ID maps guardados: ../models/id_maps.pkl
 Model info guardada: models/model_info.json
 User-tier mapping guardada: data/final/user_tier_mapping.csv
